In [1]:
''' This does all the prep for making a BlendedMLP: it splits a mesh into patches
using geodesics, and parametrises the patches using Tutte's embedding. '''


import trimesh
import open3d as o3d
import numpy as np

from scipy.spatial import cKDTree
from mesh_processing import *
import matplotlib.pyplot as plt

import os
import shutil
from visuals import *

original_cwd = os.getcwd()

In [2]:
# Settings
shape_name =   'sphere1'    #'sphere4' #'torus4608' #'sphere4'   #'sphere2' #'camelhead'
target_number_of_triangles = 6

In [3]:
''' Load mesh (fine_mesh) and make a simplified mesh (coarse_mesh) '''

# Paths
off_filepath = f"data/surfaces/{shape_name}.off"
obj_filepath = f"data/surfaces/{shape_name}.obj"

# Check if .off file exists
if not os.path.exists(off_filepath):
    if os.path.exists(obj_filepath):
        print(f"{off_filepath} not found. Converting from .obj...")
        fine_mesh = o3d.io.read_triangle_mesh(obj_filepath)
        fine_mesh.vertex_normals = o3d.utility.Vector3dVector()  # Clear vertex normals
        fine_mesh.triangle_normals = o3d.utility.Vector3dVector()  # Clear triangle normals
        o3d.io.write_triangle_mesh(off_filepath, fine_mesh)
    else:
        raise FileNotFoundError(f"Neither {off_filepath} nor {obj_filepath} found.")

# Load mesh from .off
fine_mesh = o3d.io.read_triangle_mesh(off_filepath)

# Simplify mesh
simplified_mesh = fine_mesh.simplify_quadric_decimation(
    target_number_of_triangles=target_number_of_triangles
)

# Visualize meshes
o3d.visualization.draw_geometries([fine_mesh], window_name="Fine Mesh")
o3d.visualization.draw_geometries([simplified_mesh], window_name="Simplified Mesh")

n_triangles = np.asarray(simplified_mesh.triangles).shape[0]

patches_name = shape_name + '_'+str(n_triangles)+'patch'


os.makedirs('patches/'+patches_name, exist_ok=True)
o3d.io.write_triangle_mesh('patches/'+patches_name+'/fine.obj', fine_mesh)

V_fine = np.asarray(fine_mesh.vertices)
V_simp = np.asarray(simplified_mesh.vertices)

# Snap the simplified mesh vertices to the fine mesh vertices
coarse_vertex_indices = cKDTree(V_fine).query(V_simp)[1]
V_coarse = V_fine[coarse_vertex_indices]

# Create new mesh with snapped vertices
coarse_mesh = o3d.geometry.TriangleMesh(
    vertices=o3d.utility.Vector3dVector(V_coarse),
    triangles=simplified_mesh.triangles
)
coarse_mesh.compute_vertex_normals()

o3d.visualization.draw_geometries([coarse_mesh], window_name="Coarse Mesh")

o3d.io.write_triangle_mesh("patches/"+patches_name+"/coarse.obj", coarse_mesh)


[Open3D WARNING] Write OBJ can not include triangle normals.


True

In [4]:
# Make halfedge meshes, including a halfedge index for each vertex.
coarse_he_mesh, coarse_V_he = he_mesh_from_o3d(coarse_mesh)
fine_he_mesh, fine_V_he = he_mesh_from_o3d(fine_mesh)

Preparing V_he for halfedge mesh
Prepared V_he for halfedge mesh.

Preparing V_he for halfedge mesh
Prepared V_he for halfedge mesh.



In [5]:
'''Find geodesics corresponding to coarse mesh vertices, and cut the fine mesh along the geodesics. '''

geodesic_vtx_indices = dict()
geodesic_pcds = []
half_edges = []

count = 1
total=len(coarse_he_mesh.half_edges)//2

#os.system('cp data/surfaces/'+shape_name+'.off ../geodesic_matlab/data/'+shape_name+'.off' )
#convert_off_file('../geodesic_matlab/data/'+shape_name+'.off', '../geodesic_matlab/data/'+shape_name+'.txt')


bdry_halfedge_mask = -1 * np.ones(len(fine_he_mesh.half_edges), dtype=int)



for i, he in enumerate(coarse_he_mesh.half_edges):

    v_id_start = coarse_vertex_indices [he.vertex_indices[0]]
    v_id_end = coarse_vertex_indices [he.vertex_indices[1]]

    left_face_id = he.triangle_index
    right_face_id = coarse_he_mesh.half_edges[he.twin].triangle_index

    if not ((v_id_start, v_id_end) in half_edges): # don't cut a halfedge if we already cut its twin
        
        print('Cutting the mesh along curve',str(count),'of',str(total)+'.')
        half_edges += [(v_id_start, v_id_end), (v_id_end, v_id_start)]
    
        print('start:', v_id_start, 'end:', v_id_end)


        # prepare .txt file
        fine_mesh = o3d_mesh_from_he_mesh(fine_he_mesh)
        o3d.io.write_triangle_mesh('../geodesic_matlab/data/temp.off', fine_mesh)
        o3d.io.write_triangle_mesh("patches/"+patches_name+"/fine.obj", fine_mesh)
        
        convert_off_file('../geodesic_matlab/data/temp.off', '../geodesic_matlab/data/temp.txt')
    
        # Call geodesic_matlab find_path
        os.chdir(original_cwd)
        os.chdir('../geodesic_matlab/src')  # or wherever `geodesic_matlab` is accessible from
        os.system('./find_path.out ../data/temp.txt '+str(v_id_end)+' '+str(v_id_start))
        os.chdir(original_cwd)
        
        geodesic_filepath = '../geodesic_matlab/paths/path.obj'
        os.system('cp ../geodesic_matlab/paths/path.obj patches/'+patches_name+'/path_'+str(i)+'.obj')
    
        fine_he_mesh, fine_V_he, geodesic_pcd, vtx_indices, bdry_halfedge_mask = cut_mesh_along_geodesic_v2(fine_he_mesh, fine_V_he,
                                                                                geodesic_filepath,
                                                                            bdry_halfedge_mask, v_id_start, v_id_end,
                                                                            left_face_id=left_face_id, 
                                                                            right_face_id=right_face_id, debug_plot=False )

        geodesic_vtx_indices[i] = vtx_indices
        geodesic_vtx_indices[he.twin] = vtx_indices[::-1]
    
        geodesic_pcds.append(geodesic_pcd)
        print(vtx_indices)

        #o3d.visualization.draw_geometries([he_mesh, geodesic_pcd], window_name="Cut along a curve successfully.")
    
        o3d.visualization.draw_geometries([fine_he_mesh]+geodesic_pcds, window_name="Mesh Cutting Progress "+str(count)+" of "+str(total)+".")
    
        count+=1
        
fine_mesh = o3d_mesh_from_he_mesh(fine_he_mesh)
o3d.io.write_triangle_mesh("patches/"+patches_name+"/fine.obj", fine_mesh)

#o3d.visualization.draw_geometries([fine_he_mesh]+geodesic_pcds, window_name="Final Cut Mesh")
V_fine = np.asarray(fine_he_mesh.vertices)

Cutting the mesh along curve 1 of 9.
start: 15 end: 32
Processed file saved as ../geodesic_matlab/data/temp.txt

mesh has 42 vertices, 80 faces, 120 edges
0 edges are boundary edges
shortest/longest edges are 0.546533/0.618034 = 0.884309
enclosing XYZ box: X[-1,1] Y[-1,1] Z[-1,1]
approximate diameter of the mesh is 3.4641
min/max face angles are 55.569/68.862 degrees

Path saved to ../paths/path.obj
Starting the cutting for curve number
Making a cut...
End of cutting section.
1 remaining
Cut along a curve successfully
[15, 42, 43, 44, 45, 32]
Cutting the mesh along curve 2 of 9.
start: 32 end: 11
Processed file saved as ../geodesic_matlab/data/temp.txt

mesh has 46 vertices, 88 faces, 132 edges
0 edges are boundary edges
shortest/longest edges are 0.100524/0.618034 = 0.162651
enclosing XYZ box: X[-1,1] Y[-1,1] Z[-1,1]
approximate diameter of the mesh is 3.4641
min/max face angles are 8.86194/120 degrees

Path saved to ../paths/path.obj
Starting the cutting for curve number
Making a cut

In [6]:
'''
# Debugging vertex indices
half_edges = coarse_he_mesh.half_edges
i=5

g1 = geodesic_vtx_indices[i]
g2 = geodesic_vtx_indices[half_edges[i].next]
g3 = geodesic_vtx_indices[half_edges[half_edges[i].next].next]

plt.plot(V_fine[g1][:,0], V_fine[g1][:,1], alpha=0.2)
plt.plot(V_fine[g2][:,0], V_fine[g2][:,1], alpha=0.2)
plt.plot(V_fine[g3][:,0], V_fine[g3][:,1], alpha=0.2)
plt.show()

print(g1)
print(g2)
print(g3)
'''


'\n# Debugging vertex indices\nhalf_edges = coarse_he_mesh.half_edges\ni=5\n\ng1 = geodesic_vtx_indices[i]\ng2 = geodesic_vtx_indices[half_edges[i].next]\ng3 = geodesic_vtx_indices[half_edges[half_edges[i].next].next]\n\nplt.plot(V_fine[g1][:,0], V_fine[g1][:,1], alpha=0.2)\nplt.plot(V_fine[g2][:,0], V_fine[g2][:,1], alpha=0.2)\nplt.plot(V_fine[g3][:,0], V_fine[g3][:,1], alpha=0.2)\nplt.show()\n\nprint(g1)\nprint(g2)\nprint(g3)\n'

In [7]:
''' Find all the halfedges that are part of a patch boundary (1st vertex, 2nd vertex) '''

'''
coarse_half_edges = coarse_he_mesh.half_edges
fine_half_edges = fine_he_mesh.half_edges

all_bdry_halfedges = []

for i, he in enumerate(coarse_half_edges):

    g1 = geodesic_vtx_indices[i]
    g2 = geodesic_vtx_indices[he.next]
    g3 = geodesic_vtx_indices[coarse_half_edges[he.next].next]
    
    bdry_vertices = g1[:-1]+g2[:-1]+g3[:] # why??? bug.

    bdry_halfedges = list(zip(bdry_vertices, bdry_vertices[1:] + [bdry_vertices[0]]))

    #plt.plot(V_fine[bdry_vertices][:,0],V_fine[bdry_vertices][:,1],  alpha=0.2)
    #plt.show()

    all_bdry_halfedges = all_bdry_halfedges + bdry_halfedges
    
    #plt.plot(V_fine[g1][:,0], V_fine[g1][:,1], alpha=0.2)
    #plt.plot(V_fine[g2][:,0], V_fine[g2][:,1], alpha=0.2)
    #plt.plot(V_fine[g3][:,0], V_fine[g3][:,1], alpha=0.2)
    #plt.show()
'''




'\ncoarse_half_edges = coarse_he_mesh.half_edges\nfine_half_edges = fine_he_mesh.half_edges\n\nall_bdry_halfedges = []\n\nfor i, he in enumerate(coarse_half_edges):\n\n    g1 = geodesic_vtx_indices[i]\n    g2 = geodesic_vtx_indices[he.next]\n    g3 = geodesic_vtx_indices[coarse_half_edges[he.next].next]\n    \n    bdry_vertices = g1[:-1]+g2[:-1]+g3[:] # why??? bug.\n\n    bdry_halfedges = list(zip(bdry_vertices, bdry_vertices[1:] + [bdry_vertices[0]]))\n\n    #plt.plot(V_fine[bdry_vertices][:,0],V_fine[bdry_vertices][:,1],  alpha=0.2)\n    #plt.show()\n\n    all_bdry_halfedges = all_bdry_halfedges + bdry_halfedges\n    \n    #plt.plot(V_fine[g1][:,0], V_fine[g1][:,1], alpha=0.2)\n    #plt.plot(V_fine[g2][:,0], V_fine[g2][:,1], alpha=0.2)\n    #plt.plot(V_fine[g3][:,0], V_fine[g3][:,1], alpha=0.2)\n    #plt.show()\n'

In [16]:
coarse_half_edges = coarse_he_mesh.half_edges
fine_half_edges = fine_he_mesh.half_edges

V = np.asarray(fine_he_mesh.vertices)
F = np.asarray(fine_he_mesh.triangles)

print(len(fine_half_edges))
print(bdry_halfedge_mask.shape)

# Step 3: Create a binary mask (example: highlight half of them)

#highlight_mask = (bdry_halfedge_mask>=0)

highlight_mask = (bdry_halfedge_mask==0)

# Step 4: Create geometry to visualize highlighted halfedges
highlighted_lines = []
colors = []
line_points = []

for he_id in range(len(fine_half_edges)):
    
    if highlight_mask[he_id]:
       
        idx = len(line_points)
        
        start_id, end_id = fine_half_edges[he_id].vertex_indices
        #print(start_id,end_id)
        
        line_points.extend([V[start_id,:], V[end_id,:]])
        highlighted_lines.append([idx, idx + 1])
        colors.append([1, 0, 0])  # red lines

# Create line set for highlighted halfedges
line_set = o3d.geometry.LineSet(
    points=o3d.utility.Vector3dVector(line_points),
    lines=o3d.utility.Vector2iVector(highlighted_lines),
)
line_set.colors = o3d.utility.Vector3dVector(colors)

# Step 5: Visualize mesh with highlighted halfedges
o3d.visualization.draw_geometries([ line_set])




444
(444,)


In [10]:
''' Do floodfill to find the patches, given their boundaries. '''


import mesh_processing
import importlib
importlib.reload(mesh_processing)
from mesh_processing import *


coarse_half_edges = coarse_he_mesh.half_edges
fine_half_edges = fine_he_mesh.half_edges

V = np.asarray(fine_he_mesh.vertices)
F = np.asarray(fine_he_mesh.triangles)

selected_faces = []
seed_he_index = 0

union_mask = np.array([False for i in range(len(F))])

patch_meshes = []



N = len(fine_half_edges)

while not np.all(union_mask): #while we have not accounted for all the faces
    selected_faces = floodfill(seed_he_index, fine_half_edges, bdry_halfedge_mask, V)

    mask = np.array([i in selected_faces for i in range(len(F))]) #A is region inside
    #print(mask)

    if np.any(mask & ~union_mask): #does mask instroduce any new faces?
        F_part = F[mask]
        
        union_mask += mask
    
        # Create new mesh
        A = o3d.geometry.TriangleMesh()
        A.vertices = o3d.utility.Vector3dVector(V)
        A.triangles = o3d.utility.Vector3iVector(F_part)
        A.paint_uniform_color(np.random.random(3)) # assign a colour
        A.compute_vertex_normals()

        #o3d.visualization.draw_geometries([A])

        A.remove_unreferenced_vertices()
        patch_meshes.append(A)
        
    seed_he_index = np.random.randint(N) #choose a new seed. this can definitely be more optimal.
    
        
o3d.visualization.draw_geometries(patch_meshes)

#####################################################



    

[Open3D WARNING] Write OBJ can not include triangle normals.
[Open3D WARNING] Write OBJ can not include triangle normals.
[Open3D WARNING] Write OBJ can not include triangle normals.
[Open3D WARNING] Write OBJ can not include triangle normals.
[Open3D WARNING] Write OBJ can not include triangle normals.
[Open3D WARNING] Write OBJ can not include triangle normals.


In [ ]:

''' Which patch corresponds to which coarse triangle? '''

#patch_centroids = np.array( [np.asarray(patch.vertices).mean(0) for patch in patch_meshes] )


#coarse_F = np.asarray( coarse_he_mesh.triangles )
#coarse_V = np.asarray( coarse_he_mesh.vertices )

#coarse_centroids = np.array( [ coarse_V[coarse_F[i]].mean(0) for i in range(coarse_F.shape[0]) ] )
#print(patch_centroids.shape)
#print(coarse_centroids.shape)

#P = patch_centroids
#C = coarse_centroids
#P = P - P.mean()
#C = C - C.mean()

#M = P @ C.transpose(1,0) # correlation matrix
#order = M.argmax(-1)
    
'''
# Testing that the patch-ordering is correct
for i,A in enumerate(patch_meshes):
    V_coarse = coarse_V[coarse_F[order[i]]]
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(V_coarse)

    o3d.visualization.draw_geometries([A, pcd])
'''

#for i, A in enumerate(patch_meshes):
#    triangle_id = order[i]
#    o3d.io.write_triangle_mesh("patches/"+patches_name+"/"+str(triangle_id)+".obj", A)

In [11]:
''' Add some special extra data to the patch obj files (the indices that correspond to corners) '''

# Loop over each mesh in `patch_meshes`
for i in range(n_triangles):

    A = o3d.io.read_triangle_mesh("patches/"+patches_name+"/"+str(i)+".obj")
    

    # Get the 3 vertices (abc) from coarse mesh
    abc = coarse_V[coarse_F[i]]  # `abc` should be a (3, 3) array

    # Convert mesh A's vertices to numpy array
    V = np.asarray(A.vertices)
    F = np.asarray(A.triangles)

    # Build a cKDTree for fast nearest-neighbor search in mesh A's vertices
    kdtree = cKDTree(V)

    # Find closest indices in V for each vertex in abc
    dist, closest_indices = kdtree.query(abc)  # The result should be an array of shape (3,) with the indices

    print('dist', dist)
    # Create the comment with the special indices
    comment = f"# Coarse Triangle Correspondence: {closest_indices[0]}, {closest_indices[1]}, {closest_indices[2]}"

    # Save the mesh to a temporary .obj file using Open3D
    obj_file_path = f"patches/{patches_name}/{i}.obj"
    o3d.io.write_triangle_mesh(obj_file_path, A)

    # Open the file and insert the comment at the top
    with open(obj_file_path, "r") as file:
        lines = file.readlines()

    # Prepend the comment line to the file content
    lines.insert(0, comment + "\n")

    # Write the content back to the file
    with open(obj_file_path, "w") as file:
        file.writelines(lines)

    print(f"Triangle {i}: Closest indices in V for abc: {closest_indices}")
    print(f"Mesh saved at: {obj_file_path} with special indices as comments.")

dist [0.00000000e+00 0.00000000e+00 2.45756246e-07]
Triangle 0: Closest indices in V for abc: [ 2 13 23]
Mesh saved at: patches/sphere1_6patch/0.obj with special indices as comments.
dist [0. 0. 0.]
Triangle 1: Closest indices in V for abc: [13 15 10]
Mesh saved at: patches/sphere1_6patch/1.obj with special indices as comments.
dist [0.00000000e+00 2.45756246e-07 0.00000000e+00]
Triangle 2: Closest indices in V for abc: [ 0 10  4]
Mesh saved at: patches/sphere1_6patch/2.obj with special indices as comments.
dist [2.45756246e-07 2.45756246e-07 0.00000000e+00]
Triangle 3: Closest indices in V for abc: [ 3 14  6]
Mesh saved at: patches/sphere1_6patch/3.obj with special indices as comments.
dist [0.00000000e+00 2.45756246e-07 2.45756246e-07]
Triangle 4: Closest indices in V for abc: [9 7 6]
Mesh saved at: patches/sphere1_6patch/4.obj with special indices as comments.
dist [0.00000000e+00 2.45756246e-07 0.00000000e+00]
Triangle 5: Closest indices in V for abc: [ 3 15  0]
Mesh saved at: patc

In [12]:
''' Finally, parametrise the triangles and save as {i}_param.obj '''


import importlib
import visuals
importlib.reload(visuals)

from visuals import *

import mesh_processing
importlib.reload(mesh_processing)

from mesh_processing import *


for i in range(n_triangles):
    input_filename = "patches/"+patches_name+"/"+str(i)+".obj"
    output_filename = "patches/"+patches_name+"/"+str(i)+"_param.obj"
    tutte_embedding(input_filename, output_filename, visualise=False)

boundary: [2, 20, 3, 22, 21, 13, 26, 24, 8, 29, 28, 14, 30, 23, 7, 5, 17]
[2, 13, 23]
Tutte-style harmonic embedding completed. UVs saved to patches/sphere1_6patch/0_param.obj
boundary: [0, 12, 13, 16, 6, 5, 14, 15, 20, 19, 18, 17, 10, 8, 2]
[13, 15, 10]
Tutte-style harmonic embedding completed. UVs saved to patches/sphere1_6patch/1_param.obj
boundary: [0, 14, 13, 12, 10, 6, 11, 9, 4, 18, 17, 16, 15]
[0, 10, 4]
Tutte-style harmonic embedding completed. UVs saved to patches/sphere1_6patch/2_param.obj
boundary: [0, 3, 10, 16, 13, 14, 15, 12, 11, 9, 8, 7, 5, 6, 4, 2]
[3, 14, 6]
Tutte-style harmonic embedding completed. UVs saved to patches/sphere1_6patch/3_param.obj
boundary: [0, 8, 9, 12, 11, 10, 7, 15, 14, 13, 6, 2]
[9, 7, 6]
Tutte-style harmonic embedding completed. UVs saved to patches/sphere1_6patch/4_param.obj
boundary: [0, 11, 10, 9, 8, 3, 18, 17, 16, 15, 14, 13, 12]
[3, 15, 0]
Tutte-style harmonic embedding completed. UVs saved to patches/sphere1_6patch/5_param.obj
